# Baseline Model Evaluation
This notebook evaluates the pre-trained `SamLowe/roberta-base-go_emotions` model to establish a strong baseline for our project. We can later use these results to compare against our own custom-trained models.

In [9]:
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

# Add the source folder to sys.path so we can import our modules
source_code_path = os.path.abspath('./source')
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import load_dataset
from baseline_roberta import apply_baseline_to_dataframe
from general_preprocessing import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Load the Dataset

In [10]:
# Load the dataset using the utility function
dataset_original = load_dataset('GoEmotions.csv')

# Display the first few rows to verify it loaded correctly
dataset_original.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [11]:
dataset = dataset_original.copy()

## <font color='#BFD72F' size=6>2 Modeling</font> <a class="anchor" id="4"></a>
  
[Back to TOC](toc)

After exploring the dataset, and having our pipeline for preprocessing created, we can move on to actually testing different models and understand how well they can predict the emotions of the texts.

To understand how diferent preprocessing techniques affect the performance of the model, we will apply three different preprocessings, and use these 3 versions of the text throughout the rest of the project.

### <font color='#BFD72F' size=6>2.1 Creating preprocessed versions</font> <a class="anchor" id="2.1"></a>
  
[Back to TOC](toc)

In [12]:
# Text with translated emojis, hastags, urls (basically the minimal preprocessing to be done, that the allows the model to fully understand every component of the text, without taking the 'emotion' out of it. ) 
dataset['01_minimal_preprocessing'] = dataset['text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_emojis=True,             
        no_hashtags=True,           
        hashtag_retain_words=True,  
        no_newlines=True,           
        no_urls=True,
        no_punctuation=False,  
        no_stopwords=False,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=False,
        lemmatized=False,
        list_pos=[],
        pos_tags_list='no_pos',
        tokenized_output=False,
        stemmed=False,
        treat_repeated_chars=False
    )
)

In [13]:
# Medium Preprocessing: Text with translated emojis, hastags, urls, punctuation removed, stopwords removed, diacritics converted, lowercased, lemmatized, and repeated characters treated
dataset['02_medium_preprocessing'] = dataset['text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_emojis=True,             
        no_hashtags=True,           
        hashtag_retain_words=True,  
        no_newlines=True,           
        no_urls=True,
        no_punctuation=True,  
        no_stopwords=False,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=True,
        lemmatized=False,
        list_pos=[],
        pos_tags_list='no_pos',
        tokenized_output=False,
        stemmed=False,
        treat_repeated_chars=False
    )
)

In [14]:
# Harsh Preprocessing: Text with translated emojis, hastags, urls, punctuation removed, stopwords removed, diacritics converted, lowercased, lemmatized, and repeated characters treated
dataset['03_harsh_preprocessing'] = dataset['text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_emojis=True,             
        no_hashtags=True,           
        hashtag_retain_words=True,  
        no_newlines=True,           
        no_urls=True,
        no_punctuation=True,  
        no_stopwords=True,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=True,
        lemmatized=False,
        list_pos=[],
        pos_tags_list='no_pos',
        tokenized_output=False,
        stemmed=False,
        treat_repeated_chars=False
    )
)

In [16]:
# Follow an example
print("Original Text Sample:\n")
print(dataset[['text']].iloc[818].values)

print("\nMinimal Preprocessing Sample:\n")
print(dataset[['01_minimal_preprocessing']].iloc[818].values)

print("\nMedium Preprocessing Sample:\n")
print(dataset[['02_medium_preprocessing']].iloc[818].values)

print("\nHarsh Preprocessing Sample:\n")
print(dataset[['03_harsh_preprocessing']].iloc[818].values)

Original Text Sample:

['You have another new follower 😊 love your work!']

Minimal Preprocessing Sample:

['You have another new follower emoji_smiling_face_with_smiling_eyes love your work!']

Medium Preprocessing Sample:

['you have another new follower emoji_smiling_face_with_smiling_eyes love your work']

Harsh Preprocessing Sample:

['another new follower emoji_smiling_face_with_smiling_eyes love work']


It is expected that the more preprocessing we do the worse the model will perform sibnce this tasks constitutes in classifing the emotions of the text, wgich is normally deapily correlated with punctuation, stopwords, and repeated characters. 

### 2. Apply the Baseline Model
Since evaluating all 200,000+ rows takes a significant amount of time, we first test it on a small sample of 100 rows.

In [17]:
# Test the baseline model on the first 100 rows for the minimal preprocessing version of the text
test_data = dataset.head(100)

df_with_predictions = apply_baseline_to_dataframe(
    df=test_data, 
    text_column='01_minimal_preprocessing',  
    threshold=0.5       
)

# Display the results: comparing the original text with our baseline predictions
df_with_predictions[['text', '01_minimal_preprocessing', '01_baseline_predictions']].head(10)

Downloading/Loading model 'SamLowe/roberta-base-go_emotions' into memory...


Device set to use mps:0


Gathering predictions for 100 rows... This might take a moment!


Generating predictions: 100%|██████████| 100/100 [00:09<00:00, 10.51it/s]


Predictions completed successfully!


KeyError: "['01_baseline_predictions'] not in index"

In [ ]:
# Test the baseline model on the first 100 rows for the medium preprocessing version of the text
test_data = dataset.head(100)

df_with_predictions = apply_baseline_to_dataframe(
    df=test_data, 
    text_column='02_medium_preprocessing',  
    threshold=0.5       
)

# Display the results: comparing the original text with our baseline predictions
df_with_predictions[['text', '02_medium_preprocessing', '02_baseline_predictions']].head(10)

Downloading/Loading model 'SamLowe/roberta-base-go_emotions' into memory...


Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# Test the baseline model on the first 100 rows for the harsh preprocessing version of the text
test_data = dataset.head(100)

df_with_predictions = apply_baseline_to_dataframe(
    df=test_data, 
    text_column='03_harsh_preprocessing',  
    threshold=0.5       
)

# Display the results: comparing the original text with our baseline predictions
df_with_predictions[['text', '03_harsh_preprocessing', '03_baseline_predictions']].head(10)